# 03: Fine-tuning QLoRA de MedGemma (extension Could)

> **Prototype pédagogique. Non destiné au diagnostic.** Le fine-tuning ne change pas ce statut.

Extension du niveau **Could** : adapter `google/medgemma-4b-it` par **QLoRA**
(4-bit + LoRA) pour produire le JSON du projet avec la bonne classe.

## Où s'exécute l'entraînement ?

**En cloud (Kaggle T4 16 Go)**, pas en local : une GTX 1660 (6 Go, Turing, sans
bf16 natif, fp16 buggé en génération) est trop juste pour entraîner un modèle 4B
multimodal. Le notebook d'entraînement complet est
`finetuning/medgemma_qlora_kaggle.ipynb` ; le pas-à-pas est dans
`finetuning/README.md`.

Ce notebook-ci sert à **préparer et inspecter les données** localement, et à
documenter la boucle méthodologique.

In [1]:
import sys, json, subprocess
from pathlib import Path

# Racine du dépôt, quel que soit le dossier de lancement du kernel.
ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'src').is_dir() and (candidate / 'prompts').is_dir():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

# Génère finetuning/data/train.jsonl et val.jsonl (paires image -> JSON cible).
# Étape LÉGÈRE (préparation de données), sans GPU. L'entraînement, lui, se fait sur Kaggle.
builder = ROOT / 'finetuning' / 'build_training_data.py'
assert builder.exists(), f"Script introuvable : {builder}"
res = subprocess.run([sys.executable, str(builder), '--per-class', '200', '--val-frac', '0.15'],
                     cwd=ROOT, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('ATTENTION : la generation a echoue (code', res.returncode, ') :')
    print(res.stderr[-2000:])
    print("Astuce : preparez d'abord les cas RSNA ->  python data/prepare_rsna.py --per-class 200")


ATTENTION : la generation a echoue (code 1 ) :
Traceback (most recent call last):
  File "c:\I1\S2\MasterCamp\Projet-ARVI-main\finetuning\build_training_data.py", line 137, in <module>
    main()
    ~~~~^^
  File "c:\I1\S2\MasterCamp\Projet-ARVI-main\finetuning\build_training_data.py", line 133, in main
    build(args.per_class, args.val_frac, args.seed)
    ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\I1\S2\MasterCamp\Projet-ARVI-main\finetuning\build_training_data.py", line 108, in build
    picked = select(per_class, seed)
  File "c:\I1\S2\MasterCamp\Projet-ARVI-main\finetuning\build_training_data.py", line 85, in select
    with METADATA.open(newline="", encoding="utf-8") as f:
         ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\natha\anaconda3\Lib\pathlib\_local.py", line 537, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2

In [3]:
# Inspecter un exemple d'entraînement (si la génération a réussi)
train_jsonl = ROOT / 'finetuning' / 'data' / 'train.jsonl'
if train_jsonl.exists():
    lines = train_jsonl.read_text(encoding='utf-8').splitlines()
    print('exemples train :', len(lines))
    ex = json.loads(lines[0])
    print('image :', ex['image_path'])
    print('label :', ex['label'])
    print('cible :')
    print(json.dumps(json.loads(ex['target']), indent=2, ensure_ascii=False))
else:
    print("finetuning/data/train.jsonl introuvable: exécutez d'abord la cellule de génération ci-dessus.")

finetuning/data/train.jsonl introuvable: exécutez d'abord la cellule de génération ci-dessus.


## Boucle méthodologique (à défendre en soutenance)

1. **Cibles = distillation du label RSNA** dans le schéma JSON (justifications
   gabarits, fondées sur le visible, sans invention d'historique). Ce n'est pas
   une annotation d'expert : limite à assumer.
2. **QLoRA** (adaptateur, pas de fine-tuning complet), tour de vision gelée.
3. **Évaluer l'adaptateur avec le MÊME harnais** que la baseline
   (`eval/run_evaluation.py` avec `MEDGEMMA_ADAPTER`), et comparer accuracy,
   sensibilité, spécificité, validité JSON, hallucinations.
4. **Conserver la baseline zero-shot** comme point de comparaison : un fine-tuning
   n'est une amélioration que s'il est mesuré et gagnant.

Une fois l'adaptateur entraîné sur Kaggle et téléchargé dans `finetuning/adapter/` :

```bash
$env:MEDGEMMA_ADAPTER = 'finetuning/adapter'
python eval/run_evaluation.py --engine medgemma --mode baseline \
  --cases data/rsna_cases.csv --out-dir eval/results_ft --db-path eval/results_ft/runs.sqlite
```